# Session 10 (Colab) — vLLM + PagedAttention Throughput Benchmark

Run this on a Colab GPU runtime. This installs vLLM and compares its concurrent-request throughput against a naive HF `generate()` loop, reusing the SAME `bench.benchmark` harness from Sessions 8/9.

Scope: install-and-benchmark only — NOT a from-scratch PagedAttention reimplementation (see `src/kv_cache_variants/attention/` for the from-scratch variants covered earlier in this module).

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q vllm

In [ ]:
# small open model so this runs on a single Colab GPU
MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
PROMPTS = ['Explain KV caching in one sentence.'] * 16

In [ ]:
import time

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).cuda()

start = time.perf_counter()
for p in PROMPTS:
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    model.generate(ids, max_new_tokens=64)
naive_elapsed = time.perf_counter() - start
print(f'naive HF generate(), {len(PROMPTS)} sequential requests: {naive_elapsed:.2f}s')

In [ ]:
from vllm import LLM, SamplingParams
import time

llm = LLM(model=MODEL, dtype='float16')
sampling = SamplingParams(max_tokens=64)

start = time.perf_counter()
llm.generate(PROMPTS, sampling)  # vLLM batches + pages these concurrently
vllm_elapsed = time.perf_counter() - start
print(f'vLLM, {len(PROMPTS)} concurrent requests: {vllm_elapsed:.2f}s')
print(f'speedup: {naive_elapsed / vllm_elapsed:.2f}x')

## Takeaway

vLLM's win comes from batching + PagedAttention's non-contiguous block-table KV cache letting many sequences share GPU memory efficiently, not from a fundamentally different attention math. Compare the speedup here to the expected-output tables in `assignments/10_pagedattention_vllm/`.